In [4]:
%pip install rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

# chunks + embeddings saved by rag.ipynb (row i of chunk_embs = chunks[i])
chunks = json.loads(Path("data/chunks.json").read_text(encoding="utf-8"))
chunk_embs = np.load("data/chunk_embs.npy")
print(len(chunks), "chunks,", chunk_embs.shape, "embeddings")

c:\Users\Satyajit\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1035 chunks, (1035, 384) embeddings


In [6]:
# same models as rag.ipynb
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def retrieve(query, k=10):
    """Dense only (Pipeline 1) - same as rag.ipynb."""
    t0 = time.perf_counter()
    q = bi_encoder.encode(query, normalize_embeddings=True)
    scores = chunk_embs @ q
    top_idx = np.argsort(-scores)[:k]
    elapsed = time.perf_counter() - t0
    results = [{"rank": r + 1, "chunk_id": chunks[i]["chunk_id"],
                "score": float(scores[i]), "text": chunks[i]["text"]}
               for r, i in enumerate(top_idx)]
    return results, elapsed


def rerank(query, candidates, k=3):
    """Cross-encoder re-ranking (Pipeline 2) - same as rag.ipynb."""
    t0 = time.perf_counter()
    ce_scores = cross_encoder.predict([(query, c["text"]) for c in candidates])
    elapsed = time.perf_counter() - t0
    reranked = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)
    for new_rank, c in enumerate(reranked, 1):
        c["new_rank"] = new_rank
    return reranked[:k], reranked, elapsed

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3122.44it/s]


In [7]:
STOPWORDS = set("""a an the of and or to in on for by with is are be shall any
such as that this it its at from under which who whom what can does do not""".split())

def tokenize(text):
    # lowercase words and numbers; "21A" -> "21a", so article numbers stay intact
    return [t for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in STOPWORDS]

# build the BM25 index once: one token list per chunk (same order as chunks / chunk_embs)
bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks], k1=1.5, b=0.75)

# quick look: which chunks does BM25 alone rank highest?
q = "What does Article 31 say?"
print(tokenize(q))
[chunks[i]["chunk_id"] for i in np.argsort(-bm25.get_scores(tokenize(q)))[:5]]

['article', '31', 'say']


['31_0', '366_0', '352_2', '69_0', '326_0']

In [8]:
def rrf_fuse(rankings, k=60):
    """Each list votes 1/(k + rank) for each chunk; chunks ranked high in either list win."""
    fused = Counter()
    for ranking in rankings:
        for rank, idx in enumerate(ranking, start=1):
            fused[idx] += 1 / (k + rank)
    return fused.most_common()          # [(chunk_index, rrf_score), ...] best first


def retrieve_hybrid(query, k=10, pool=50):
    """Hybrid retrieval - same output format as retrieve(), so rerank() works unchanged."""
    t0 = time.perf_counter()
    q = bi_encoder.encode(query, normalize_embeddings=True)
    dense_rank = np.argsort(-(chunk_embs @ q))[:pool]                    # top 50 by meaning
    bm25_rank  = np.argsort(-bm25.get_scores(tokenize(query)))[:pool]    # top 50 by keywords
    fused = rrf_fuse([dense_rank, bm25_rank])[:k]
    elapsed = time.perf_counter() - t0

    results = [{"rank": r + 1,
                "chunk_id": chunks[i]["chunk_id"],
                "score": float(s),                  # RRF score, not cosine
                "text": chunks[i]["text"]}
               for r, (i, s) in enumerate(fused)]
    return results, elapsed

In [9]:
# one query, side by side: dense top 10 vs hybrid top 10 vs hybrid + cross-encoder top 3
query = "What does Article 31 say?"

dense10, t_dense = retrieve(query)
hybrid10, t_hybrid = retrieve_hybrid(query)
hybrid_top3, hybrid_all, t_ce = rerank(query, hybrid10)

print("dense  :", [c["chunk_id"] for c in dense10], f"({t_dense*1000:.0f} ms)")
print("hybrid :", [c["chunk_id"] for c in hybrid10], f"({t_hybrid*1000:.0f} ms)")
print("hybrid + rerank top 3:", [c["chunk_id"] for c in hybrid_top3], f"({t_ce*1000:.0f} ms)")

dense  : ['31D_0', '32A_0', '31B_0', '278_0', '31A_2', '386_0', '381_0', '242_0', '36_0', '232_0'] (229 ms)
hybrid : ['31_0', '366_3', '145_2', '338B_10', '320_6', '35_1', '368_2', '323A_3', '31C_0', '31D_0'] (28 ms)
hybrid + rerank top 3: ['31_0', '31D_0', '31C_0'] (519 ms)


## Dense vs hybrid on all failure queries (retrieval only, no LLM calls)

For each query: the best rank of an expected article in four setups.

| column | pipeline |
|---|---|
| `dense@10` | bi-encoder top 10 |
| `dense+ce@3` | bi-encoder top 10 → cross-encoder → top 3 (what Pipeline 2 sends to the LLM) |
| `hybrid@10` | BM25 + dense (RRF) top 10 |
| `hybrid+ce@3` | hybrid top 10 → cross-encoder → top 3 |

`-` means not found. Results are saved to `failure_modes/results_hybrid/summary.json`.

In [10]:
QUERIES_DIR = Path("failure_modes/queries")
OUT_DIR = Path("failure_modes/results_hybrid")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def best_rank(results, expected, rank_key="rank"):
    """Best (lowest) rank at which any chunk of an expected article appears, or None."""
    ranks = [c[rank_key] for c in results if c["chunk_id"].split("_")[0] in expected]
    return min(ranks) if ranks else None


def show(r):
    return "-" if r is None else str(r)


summary = []
print(f"{'dense@10':>8} {'dense+ce@3':>10} {'hybrid@10':>9} {'hybrid+ce@3':>11}   query")
for qfile in sorted(QUERIES_DIR.glob("*.json")):
    for item in json.loads(qfile.read_text(encoding="utf-8")):
        q, expected = item["query"], item["expected_articles"]
        if not expected:                       # out-of-corpus queries have nothing to find
            continue

        dense10, t_d = retrieve(q)
        dense_top3, _, t_ce_d = rerank(q, dense10)
        hybrid10, t_h = retrieve_hybrid(q)
        hybrid_top3, _, t_ce_h = rerank(q, hybrid10)

        row = {
            "mode": qfile.stem, "query": q, "expected_articles": expected,
            "dense@10":    best_rank(dense10, expected),
            "dense+ce@3":  best_rank(dense_top3, expected, "new_rank"),
            "hybrid@10":   best_rank(hybrid10, expected),
            "hybrid+ce@3": best_rank(hybrid_top3, expected, "new_rank"),
            "dense_top3":  [c["chunk_id"] for c in dense_top3],
            "hybrid_top3": [c["chunk_id"] for c in hybrid_top3],
            "latency_ms": {"dense": round(t_d * 1000, 1), "hybrid": round(t_h * 1000, 1),
                           "rerank_dense": round(t_ce_d * 1000, 1), "rerank_hybrid": round(t_ce_h * 1000, 1)},
        }
        summary.append(row)
        print(f"{show(row['dense@10']):>8} {show(row['dense+ce@3']):>10} "
              f"{show(row['hybrid@10']):>9} {show(row['hybrid+ce@3']):>11}   {q}")

(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nsaved ->", OUT_DIR / "summary.json")

dense@10 dense+ce@3 hybrid@10 hybrid+ce@3   query
       1          1         1           1   What does Article 21 say?
       1          1         1           1   What is Article 356 about?
       2          1         1           1   What is Article 370?
       -          -         7           -   Is the right to property a fundamental right?
       -          -         1           1   What does Article 31 say?
       -          -         3           2   Can peaceful assembly be restricted in the interest of public order?
       1          3         1           3   What does the right to equality say?
       1          1         1           1   What are the fundamental duties of citizens?
       1          2         2           3   What are the Directive Principles of State Policy?
       1          1         1           1   Can the State make special laws only for women?
      10          -         -           -   Can the State reserve government jobs for backward classes?
       1  

In [11]:
# totals: Hit@k = fraction of queries where an expected article is found; MRR = mean of 1/rank (0 if missing)
def hit(col):
    return sum(r[col] is not None for r in summary) / len(summary)

def mrr(col):
    return sum(1 / r[col] for r in summary if r[col] is not None) / len(summary)

print(f"{len(summary)} queries with an expected article\n")
print(f"{'setup':<12} {'Hit':>6} {'MRR':>6}")
for col in ["dense@10", "dense+ce@3", "hybrid@10", "hybrid+ce@3"]:
    print(f"{col:<12} {hit(col):>6.2f} {mrr(col):>6.2f}")

better = [r["query"] for r in summary if r["hybrid+ce@3"] and not r["dense+ce@3"]]
worse  = [r["query"] for r in summary if r["dense+ce@3"] and not r["hybrid+ce@3"]]
print("\nhybrid fixes (in top 3 only with hybrid):", *better, sep="\n  ")
print("\nhybrid breaks (in top 3 only with dense):", *worse, sep="\n  ")

lat = {k: np.mean([r["latency_ms"][k] for r in summary[1:]]) for k in summary[0]["latency_ms"]}  # skip first: warm-up
print("\nmean latency (ms):", {k: round(v, 1) for k, v in lat.items()})

42 queries with an expected article

setup           Hit    MRR
dense@10       0.81   0.72
dense+ce@3     0.79   0.68
hybrid@10      0.90   0.73
hybrid+ce@3    0.83   0.71

hybrid fixes (in top 3 only with hybrid):
  What does Article 31 say?
  Can peaceful assembly be restricted in the interest of public order?

hybrid breaks (in top 3 only with dense):

mean latency (ms): {'dense': np.float64(28.5), 'hybrid': np.float64(28.3), 'rerank_dense': np.float64(555.5), 'rerank_hybrid': np.float64(525.0)}
